# Graph model playground (`RawModelData` → `ResolvedModelData`)

Interactive checks for the **tabular graph logistics model** in `gbp/core` and `gbp/build`:

- Schema validation on `RawModelData`
- Business validation (`validate_raw_model`)
- Full build pipeline (`build_model`)
- Inspecting edges, demand resolution, lead times, fleet capacity, spines
- Optional experiments (rule-built edges, time resolution, dict I/O)

**Run** with the project venv active and working directory either the repo root or `notebooks/` (the next cell fixes `sys.path`).

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

# Repo root on sys.path (works if cwd is repo root or notebooks/)
_here = Path.cwd().resolve()
REPO_ROOT = _here if (_here / "pyproject.toml").exists() else _here.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from gbp.build import build_model, validate_raw_model
from gbp.build.time_resolution import resolve_to_periods
from gbp.core.enums import FacilityRole, PeriodType
from gbp.core.model import RawModelData
from gbp.core.attributes.defaults import get_all_default_specs
from gbp.io import raw_to_dict

print("REPO_ROOT:", REPO_ROOT)

REPO_ROOT: E:\Documents\vlzm\GFDRR


## 1. Build a toy bike-sharing `RawModelData`

Small network: depot `d1`, stations `s1`/`s2`, commodity `working_bike`, three day periods. Pre-built **edges** and **edge_commodities** satisfy resource compatibility checks during validation.

In [2]:
from datetime import date


def make_toy_raw_model(
    *,
    with_edges: bool = True,
    with_demand: bool = True,
    with_supply: bool = False,
) -> RawModelData:
    """Minimal bike-sharing raw model (mirrors tests/unit/build/fixtures.py)."""
    facilities = pd.DataFrame(
        {
            "facility_id": ["d1", "s1", "s2"],
            "facility_type": ["depot", "station", "station"],
            "name": ["Depot", "Station 1", "Station 2"],
        }
    )
    commodity_categories = pd.DataFrame(
        {
            "commodity_category_id": ["working_bike"],
            "name": ["Working bike"],
            "unit": ["bike"],
        }
    )
    resource_categories = pd.DataFrame(
        {
            "resource_category_id": ["rebalancing_truck"],
            "name": ["Truck"],
            "base_capacity": [20.0],
            "capacity_unit": ["bike"],
        }
    )
    planning_horizon = pd.DataFrame(
        {
            "planning_horizon_id": ["h1"],
            "name": ["H1"],
            "start_date": [date(2025, 1, 1)],
            "end_date": [date(2025, 1, 4)],
        }
    )
    planning_horizon_segments = pd.DataFrame(
        {
            "planning_horizon_id": ["h1"],
            "segment_index": [0],
            "start_date": [date(2025, 1, 1)],
            "end_date": [date(2025, 1, 4)],
            "period_type": [PeriodType.DAY.value],
        }
    )
    periods = pd.DataFrame(
        {
            "period_id": ["p0", "p1", "p2"],
            "planning_horizon_id": ["h1", "h1", "h1"],
            "segment_index": [0, 0, 0],
            "period_index": [0, 1, 2],
            "period_type": [PeriodType.DAY.value] * 3,
            "start_date": [date(2025, 1, 1), date(2025, 1, 2), date(2025, 1, 3)],
            "end_date": [date(2025, 1, 2), date(2025, 1, 3), date(2025, 1, 4)],
        }
    )
    if with_supply:
        facility_roles = pd.DataFrame(
            {
                "facility_id": ["d1", "d1", "s1", "s1", "s1", "s2", "s2", "s2"],
                "role": [
                    FacilityRole.STORAGE.value,
                    FacilityRole.TRANSSHIPMENT.value,
                    FacilityRole.SOURCE.value,
                    FacilityRole.SINK.value,
                    FacilityRole.STORAGE.value,
                    FacilityRole.SINK.value,
                    FacilityRole.STORAGE.value,
                    FacilityRole.SOURCE.value,
                ],
            }
        )
    else:
        facility_roles = pd.DataFrame(
            {
                "facility_id": ["d1", "d1", "s1", "s1", "s1", "s2", "s2"],
                "role": [
                    FacilityRole.STORAGE.value,
                    FacilityRole.TRANSSHIPMENT.value,
                    FacilityRole.SINK.value,
                    FacilityRole.STORAGE.value,
                    FacilityRole.SOURCE.value,
                    FacilityRole.SINK.value,
                    FacilityRole.STORAGE.value,
                ],
            }
        )
    facility_operations = pd.DataFrame(
        {
            "facility_id": ["d1", "d1", "d1", "s1", "s1", "s1", "s2", "s2", "s2"],
            "operation_type": [
                "receiving",
                "storage",
                "dispatch",
                "receiving",
                "storage",
                "dispatch",
                "receiving",
                "storage",
                "dispatch",
            ],
            "enabled": [True] * 9,
        }
    )
    edge_rules = pd.DataFrame(
        {
            "source_type": ["depot", "depot"],
            "target_type": ["station", "station"],
            "commodity_category": ["working_bike", "working_bike"],
            "modal_type": ["road", "road"],
            "enabled": [True, True],
        }
    )
    edges = edge_commodities = None
    if with_edges:
        edges = pd.DataFrame(
            {
                "source_id": ["d1", "d1"],
                "target_id": ["s1", "s2"],
                "modal_type": ["road", "road"],
                "distance": [1.0, 2.0],
                "distance_unit": ["km", "km"],
                "lead_time_hours": [24.0, 48.0],
                "reliability": [0.99, 0.99],
            }
        )
        edge_commodities = pd.DataFrame(
            {
                "source_id": ["d1", "d1"],
                "target_id": ["s1", "s2"],
                "modal_type": ["road", "road"],
                "commodity_category": ["working_bike", "working_bike"],
                "enabled": [True, True],
                "capacity_consumption": [1.0, 1.0],
            }
        )
    demand = None
    if with_demand:
        demand = pd.DataFrame(
            {
                "facility_id": ["s1", "s1", "s2"],
                "commodity_category": ["working_bike"] * 3,
                "date": [date(2025, 1, 1), date(2025, 1, 2), date(2025, 1, 1)],
                "quantity": [1.0, 2.0, 3.0],
                "quantity_unit": ["bike"] * 3,
            }
        )
    supply = None
    if with_supply:
        supply = pd.DataFrame(
            {
                "facility_id": ["s1", "s1"],
                "commodity_category": ["working_bike", "working_bike"],
                "date": [date(2025, 1, 1), date(2025, 1, 2)],
                "quantity": [5.0, 5.0],
                "quantity_unit": ["bike", "bike"],
            }
        )
    resource_fleet = pd.DataFrame(
        {
            "facility_id": ["d1"],
            "resource_category": ["rebalancing_truck"],
            "count": [3],
        }
    )
    return RawModelData(
        facilities=facilities,
        commodity_categories=commodity_categories,
        resource_categories=resource_categories,
        planning_horizon=planning_horizon,
        planning_horizon_segments=planning_horizon_segments,
        periods=periods,
        facility_roles=facility_roles,
        facility_operations=facility_operations,
        edge_rules=edge_rules,
        edges=edges,
        edge_commodities=edge_commodities,
        demand=demand,
        supply=supply,
        resource_fleet=resource_fleet,
        resource_commodity_compatibility=pd.DataFrame(
            {
                "resource_category": ["rebalancing_truck"],
                "commodity_category": ["working_bike"],
                "enabled": [True],
            }
        ),
        resource_modal_compatibility=pd.DataFrame(
            {
                "resource_category": ["rebalancing_truck"],
                "modal_type": ["road"],
                "enabled": [True],
            }
        ),
    )


raw = make_toy_raw_model()
raw.facilities

,facility_id,facility_type,name
0,d1,depot,Depot
1,s1,station,Station 1
2,s2,station,Station 2


## 2. Column / type validation (`RawModelData.validate`)

Ensures required tables exist and columns match Pydantic row schemas in `gbp/core/schemas`.

In [3]:
raw.validate()
print("OK: RawModelData.validate() passed")

OK: RawModelData.validate() passed


## 3. Business validation (`validate_raw_model`)

Units, roles vs demand/supply, resource compatibility, connectivity, etc. Warnings use `ValidationError(level='warning')`.

In [4]:
vr = validate_raw_model(raw)
print("is_valid:", vr.is_valid, "| has_warnings:", vr.has_warnings)
for e in vr.errors:
    print(f"[{e.level}] {e.category} / {e.entity}: {e.message}")
vr.raise_if_invalid()

is_valid: True | has_warnings: True
[warning] temporal / demand: Dates may not cover full planning horizon [2025-01-01, 2025-01-04)


## 4. Full build pipeline (`build_model`)

Time resolution → edges (or use pre-built) → lead times → transformations → fleet capacity → spines.

In [5]:
resolved = build_model(raw)
type(resolved)

gbp.core.model.ResolvedModelData

## 5. Inspect resolved artifacts

Toggle columns or `head()` as you like.

In [6]:
display(resolved.periods)
print("--- demand (period_id grain) ---")
display(resolved.demand)
print("--- edges ---")
display(resolved.edges)
print("--- edge_lead_time_resolved (sample) ---")
display(resolved.edge_lead_time_resolved.head(10))
print("--- fleet_capacity ---")
display(resolved.fleet_capacity)
print("--- facility_spines keys ---", None if resolved.facility_spines is None else list(resolved.facility_spines.keys()))
print("--- edge_spines keys ---", None if resolved.edge_spines is None else list(resolved.edge_spines.keys()))
print("--- resource_spines keys ---", None if resolved.resource_spines is None else list(resolved.resource_spines.keys()))

,period_id,planning_horizon_id,segment_index,period_index,period_type,start_date,end_date
0,p0,h1,0,0,day,2025-01-01,2025-01-02
1,p1,h1,0,1,day,2025-01-02,2025-01-03
2,p2,h1,0,2,day,2025-01-03,2025-01-04


--- demand (period_id grain) ---


,facility_id,commodity_category,period_id,quantity
0,s1,working_bike,p0,1.0
1,s1,working_bike,p1,2.0
2,s2,working_bike,p0,3.0


--- edges ---


,source_id,target_id,modal_type,distance,distance_unit,lead_time_hours,reliability
0,d1,s1,road,1.0,km,24.0,0.99
1,d1,s2,road,2.0,km,48.0,0.99


--- edge_lead_time_resolved (sample) ---


,source_id,target_id,modal_type,period_id,lead_time_periods,arrival_period_id
0,d1,s1,road,p0,1,p1
1,d1,s1,road,p1,1,p2
2,d1,s1,road,p2,1,NaN
3,d1,s2,road,p0,2,p2
4,d1,s2,road,p1,2,NaN
5,d1,s2,road,p2,2,NaN


--- fleet_capacity ---


,facility_id,resource_category,effective_capacity
0,d1,rebalancing_truck,60.0


--- facility_spines keys --- ['group_0']
--- edge_spines keys --- ['group_0']
--- resource_spines keys --- ['group_0']


## 6. Experiment: edges only from `edge_rules`

Clear pre-built `edges` / `edge_commodities` and rebuild. **Note:** `validate_raw_model` may skip some checks that need pre-materialized edges; `build_model` still builds edges from rules.

In [7]:
raw_rules_only = make_toy_raw_model()
raw_rules_only.edges = None
raw_rules_only.edge_commodities = None
raw_rules_only.resource_commodity_compatibility = None
raw_rules_only.resource_modal_compatibility = None

vr2 = validate_raw_model(raw_rules_only)
print("is_valid:", vr2.is_valid, "warnings/errors count:", len(vr2.errors))
for e in vr2.errors:
    print(f"[{e.level}] {e.message}")

resolved2 = build_model(raw_rules_only)
print("Built edges:", len(resolved2.edges), "rows")
display(resolved2.edges)

is_valid: True warnings/errors count: 1
[warning] Dates may not cover full planning horizon [2025-01-01, 2025-01-04)
Built edges: 2 rows


,source_id,target_id,modal_type,distance,distance_unit,lead_time_hours,reliability
0,d1,s1,road,0.0,km,24.0,None
1,d1,s2,road,0.0,km,24.0,None


## 7. `resolve_to_periods` in isolation

Same logic as inside `resolve_all_time_varying` for date-keyed tables.

In [8]:
cost_df = pd.DataFrame(
    {
        "facility_id": ["s1", "s1", "s1"],
        "operation_type": ["dispatch"] * 3,
        "commodity_category": ["working_bike"] * 3,
        "date": [date(2025, 1, 1), date(2025, 1, 1), date(2025, 1, 2)],
        "cost_per_unit": [10.0, 12.0, 11.0],
        "cost_unit": ["EUR/bike"] * 3,
    }
)
out = resolve_to_periods(
    cost_df,
    raw.periods,
    value_columns=["cost_per_unit"],
    group_grain=["facility_id", "operation_type", "commodity_category"],
    agg_func="mean",
)
display(out)

,facility_id,operation_type,commodity_category,period_id,cost_per_unit
0,s1,dispatch,working_bike,p0,11.0
1,s1,dispatch,working_bike,p1,11.0


## 8. Default attribute specs (spine assembly)

Catalog used by `assemble_spines` when no custom list is passed.

In [9]:
specs = get_all_default_specs()
for s in specs:
    print(f"{s.entity_type:8} | {s.name:22} | grain={s.grain} | table={s.source_table}")

facility | operation_cost         | grain=('facility_id', 'operation_type', 'commodity_category', 'date') | table=operation_costs
facility | operation_capacity     | grain=('facility_id', 'operation_type', 'commodity_category') | table=operation_capacities
edge     | edge_distance          | grain=('source_id', 'target_id', 'modal_type') | table=edges
edge     | edge_lead_time_hours   | grain=('source_id', 'target_id', 'modal_type') | table=edges
edge     | edge_reliability       | grain=('source_id', 'target_id', 'modal_type') | table=edges
edge     | transport_cost         | grain=('source_id', 'target_id', 'modal_type', 'resource_category', 'date') | table=transport_costs
edge     | edge_capacity          | grain=('source_id', 'target_id', 'modal_type', 'date') | table=edge_capacities
resource | resource_base_capacity | grain=('resource_category',) | table=resource_categories
resource | resource_fixed_cost    | grain=('resource_category', 'date') | table=resource_costs
resource | re

## 9. Optional: JSON-serializable dict (small preview)

Full `raw_to_dict` can be large; here we only show top-level keys.

In [10]:
blob = raw_to_dict(raw)
print("Top-level keys:", sorted(blob.keys()))
print("facilities rows:", len(blob["facilities"]))

Top-level keys: ['commodity_categories', 'demand', 'edge_commodities', 'edge_rules', 'edges', 'facilities', 'facility_operations', 'facility_roles', 'periods', 'planning_horizon', 'planning_horizon_segments', 'resource_categories', 'resource_commodity_compatibility', 'resource_fleet', 'resource_modal_compatibility']
facilities rows: 3
